<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/results/test_split_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
data = []
with open('test_dataset.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [2]:
result_list = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']
    mentions = []
    for label in entry['annotations']:
        label_values = list(label.values())
        mentions.append(label_values)
    result_list.append({
            'patient_id': patient_id,
            'label': mentions,
            'text': text
        })

In [3]:
import pandas as pd
result_df = []

for entry in data:
    patient_id = entry['id']
    text = entry['text']

    for annotation in entry['annotations']:
        result_df.append({
            'patient_id': patient_id,
            'start': annotation['start'],
            'end': annotation['end'],
            'code': annotation['code'],
            'mention': annotation['mention']
        })
result_df = pd.DataFrame(result_df)

In [4]:
print(f"Total Annotations in the Dataset: {len(result_df)}")

patient_stats = result_df.groupby('patient_id').size().agg(['count', 'min', 'max', 'mean'])

print("Number of Patients/ EHR's:", patient_stats['count'])
print("Min Annotations per Patient:", patient_stats['min'])
print("Max Annotations per Patient:", patient_stats['max'])
print("Average Annotations per Patient:", patient_stats['mean'])

Total Annotations in the Dataset: 4626
Number of Patients/ EHR's: 500.0
Min Annotations per Patient: 1.0
Max Annotations per Patient: 32.0
Average Annotations per Patient: 9.252


In [5]:
j = 0
annotations_new = []
b_entity_count = 0
i_entity_count = 0
for i, document_data in enumerate(result_list):
    patient_id = document_data["patient_id"]
    text = document_data["text"]
    recognized_entities = []
    previous_end = -1
    for annotation in result_list[i]["label"]:
        start = annotation[0]
        end = annotation[1]
        icd_code = annotation[2]
        text1 = text
        name = text1[start:end]
        words = name.split()
        current_start = start
        for idx, word in enumerate(words):
            if idx == 0:
                type = "B-ENTITY"
                recognized_entities.append({
                "start": start,
                "end": start + len(word),
                "type": type,
                'word_name': word,
                'icd_code': icd_code })
                current_start = start + len(word)
                b_entity_count = b_entity_count + 1
        if current_start < end:
          type = "I-ENTITY"
          recognized_entities.append({
          "start": current_start,
          "end": end,
          "type": type,
          'word_name': word,
          'icd_code': icd_code })
          i_entity_count = i_entity_count + 1
          current_start = current_start + len(word)

    document_entry = {
        "patient_id": patient_id,
        "text": text,
        "recognized_entities": recognized_entities
    }
    annotations_new.append(document_entry)
print("Number of B entities:", b_entity_count)
print("Number of I entities:", i_entity_count)

Number of B entities: 4626
Number of I entities: 2491


In [6]:
%pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which i

In [9]:
from datasets import Dataset, Features, Sequence, Value, ClassLabel
tokens_list = []
ner_tags_list = []
patient_id_list = []
for j,dat in enumerate(annotations_new):
    tokens = (list(dat["text"]))
    ner_tags = ["O"] * len(tokens)
    for ent in annotations_new[j]["recognized_entities"]:
        for i in range(ent['start'], ent['end']):
            ner_tags[i] = ent['type']
    tokens_list.append(tokens)
    ner_tags_list.append(ner_tags)
    patient_id_list.append(dat['patient_id'])

features = Features({
    "tokens": Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
    "ner_tags": Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None),
    "patient_id": Value(dtype='int64', id=None)
})
final_dataset = Dataset.from_dict(
    {"tokens": tokens_list, "ner_tags": ner_tags_list,"patient_id": patient_id_list},
    features=features
)

##Split by section

In [10]:
import re
def split_text_by_titles(text, titles):
    title_pattern = re.compile('|'.join([title for title in titles]))
    start_title_positions = [match.start() for match in re.finditer(title_pattern, text)]
    title_positions = start_title_positions + [len(text)]
    sections = [text[title_positions[i]:title_positions[i+1]] for i in range(len(title_positions)-1)]
    return sections, title_positions
titles = ['ΕΝΗΜΕΡΩΤΙΚΟ ΣΗΜΕΙΩΜΑ',
'ΣΤΟΙΧΕΙΑ ΑΣΘΕΝΟΥΣ',
'ΙΣΤΟΡΙΚΟ – ΑΝΤΙΚΕΙΜΕΝΙΚΗ ΕΞΕΤΕΤΑΣΗ',
'ΠΟΡΕΙΑ ΝΟΣΟΥ',
'ΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ',
'ΘΕΡΑΠΕΥΤΙΚΗ ΑΓΩΓΗ - ΕΠΕΜΒΑΣΕΙΣ',
'ΟΔΗΓΙΕΣ ΚΑΤΑ ΤΗΝ ΕΞΟΔΟ - ΠΑΡΑΤΗΡΗΣΕΙΣ',
'ΕΡΓΑΣΤΗΡΙΑΚΕΣ ΕΞΕΤΑΣΕΙΣ',
'ΕΞΕΤΑΣΕΙΣ',
'Λοιπές εξετάσεις',
'ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΑΝΤΙΚΕΙΜΕΝΙΚΗ ΕΞΕΤΕΤΑΣΗ - ΙΣΤΟΡΙΚΟ',
'ΠΑΡΑΚΛΙΝΙΚΕΣ ΕΞΕΤΑΣΕΙΣ',
'ΙΣΤΟΡΙΚΟ – ΑΝΤΙΚΕΙΜΕΝΙΚΗ ΕΞΕΤΑΣΗ'
]

In [11]:
text = ''.join(final_dataset[0]['tokens'])
splited, title_positions = split_text_by_titles(text, titles)

In [13]:
from datasets import Features, Sequence, Value, ClassLabel, Dataset
sections_tokens_list = []
sections_ner_tags_list = []
patient_id_list = []
for i, example in enumerate(final_dataset):
    character_tokens = example['tokens']
    text = "".join(character_tokens)
    sections, title_positions =  split_text_by_titles(text,titles)
    sections_tokens_list.append([list(section) for section in sections])
    sections_ner_tags = []
    ner_tags = example['ner_tags']
    for i, section in enumerate(sections):
      start = title_positions[i]
      end = title_positions[i + 1]
      sections_ner_tags.append(ner_tags[start:end])
      if len(section) != len(ner_tags[start:end]):
          print("tokens length", len(section))
          print("ner_tags legnth:", len(ner_tags[start:end]))
          print('difference in alignment')
    sections_ner_tags_list.append(sections_ner_tags)
    patient_id_list.append(example["patient_id"])
    if len(sections_tokens_list[-1]) != len(sections_ner_tags_list[-1]):
      print(f"Example {i + 1}:")
      print("sections_tokens_list length:", len(sections_tokens_list[-1]))
      print("sections_ner_tags_list length:", len(sections_ner_tags_list[-1]))

if len(sections_tokens_list) != len(sections_ner_tags_list):
    print("Final lengths:")
    print("sections_tokens_list length:", len(sections_tokens_list))
    print("sections_ner_tags_list length:", len(sections_ner_tags_list))
    print("patient_id_list length:", len(patient_id_list))
    print()
features=Features({
    "patient_id": Value(dtype='int32', id=None),
    "sections_tokens": Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
    "sections_ner_tags": Sequence(feature=Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None), length=-1, id=None),
})
final_dataset = Dataset.from_dict(
{'sections_tokens': sections_tokens_list,'sections_ner_tags':sections_ner_tags_list, "patient_id":  patient_id_list},features=features)

In [15]:
import pickle
with open('final_test_dataset_sections.pickle', 'wb') as output:
    pickle.dump(final_dataset, output)

##Split by sentence

In [16]:
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 840.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 31.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu1

In [17]:
import stanza
stanza.download('el')
nlp_stanza = stanza.Pipeline('el')

INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: el (Greek) ...


INFO:stanza:Downloaded file to /root/stanza_resources/el/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: el (Greek):
| Processor | Package      |
----------------------------
| tokenize  | gdt          |
| mwt       | gdt          |
| pos       | gdt_nocharlm |
| lemma     | gdt_nocharlm |
| depparse  | gdt_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


In [18]:
nlp_stanza = stanza.Pipeline('el')
def tokenize_text_with_indices(text):
    doc = nlp_stanza(text)
    tokenized_sentences = []
    for sentence in doc.sentences:
        pattern = re.escape(sentence.text)
        match = re.search(pattern, text)
        if match:
            start_index = match.start()
            end_index = match.end()
            sentence_info = {
                'text': sentence.text,
                'start_index': start_index,
                'end_index': end_index
            }
            tokenized_sentences.append(sentence_info)
    return tokenized_sentences

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: el (Greek):
| Processor | Package      |
----------------------------
| tokenize  | gdt          |
| mwt       | gdt          |
| pos       | gdt_nocharlm |
| lemma     | gdt_nocharlm |
| depparse  | gdt_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


In [19]:
from datasets import Features, Sequence, Value, ClassLabel, Dataset
sentences_tokens_list = []
sentences_ner_tags_list = []
patient_id_list = []
extra_info_list = []
for j, example in enumerate(final_dataset):
    doc_sentences_ner_tags_list = []
    doc_patient_id_list = []
    doc_extra_info_list = []
    doc_sentences_tokens_list = []
    for i in range(len(example['sections_tokens'])):
      character_tokens = example['sections_tokens'][i]
      text = "".join(character_tokens)
      sentences_info =  tokenize_text_with_indices(text)
      sentences = [info['text'] for info in sentences_info]
      doc_sentences_tokens_list.append([list(sentence) for sentence in sentences])
      sentences_ner_tags = []
      ner_tags = example['sections_ner_tags'][i]

      for info in sentences_info:
          start_index = info['start_index']
          end_index = info['end_index']
          sentence_ner_tags = ner_tags[start_index:end_index]
          sentences_ner_tags.append(sentence_ner_tags)
      doc_sentences_ner_tags_list.append(sentences_ner_tags)
      if len(doc_sentences_ner_tags_list) != len(doc_sentences_tokens_list):
        print(f"Example {j+ 1}:")
        print("sentences_tokens_list length:", len(doc_sentences_ner_tags_list))
        print("sentence_ner_tags_list length:", len(doc_sentences_tokens_list))
        print('patient id ',doc_patient_id_list[-1])

      if len(doc_sentences_ner_tags_list[-1]) != len(doc_sentences_tokens_list[-1]):
        print(f"Example {j+ 1}:")
        print("sentences_tokens_list length:", len(doc_sentences_ner_tags_list[-1]))
        print("sentence_ner_tags_list length:", len(doc_sentences_tokens_list[-1]))
        print('patient id ',doc_patient_id_list[-1])
    doc_sentences_tokens_list = [item for sublist in doc_sentences_tokens_list for item in sublist]
    doc_sentences_ner_tags_list = [item for sublist in doc_sentences_ner_tags_list for item in sublist]
    if len(doc_sentences_ner_tags_list[-1]) != len(doc_sentences_tokens_list[-1]):
        print(f"Example {j+ 1}:")
        print("sentences_tokens_list length:", len(doc_sentences_ner_tags_list[-1]))
        print("sentence_ner_tags_list length:", len(doc_sentences_tokens_list[-1]))
        print('patient id ',doc_patient_id_list[-1])
    if len(doc_sentences_tokens_list) != len(doc_sentences_ner_tags_list):
        print(len(doc_sentences_tokens_list))
        print(len(doc_sentences_ner_tags_list))
        print('error')
    if len(doc_patient_id_list) != len(doc_extra_info_list):
        print('error')
    sentences_tokens_list.append(doc_sentences_tokens_list)
    sentences_ner_tags_list.append(doc_sentences_ner_tags_list)
    patient_id_list.append(example["patient_id"])
features=Features({
    "patient_id": Value(dtype='int32', id=None),
    "sentences_tokens": Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None),
    "sentences_ner_tags": Sequence(feature=Sequence(feature=ClassLabel(names=["O", "B-ENTITY",'I-ENTITY'], id=None), length=-1, id=None), length=-1, id=None),
})
final_dataset = Dataset.from_dict(
{'sentences_tokens': sentences_tokens_list,'sentences_ner_tags': sentences_ner_tags_list, "patient_id":  patient_id_list},features=features)

In [20]:
import os
with open('final_test_dataset_sentences.pickle', 'wb') as output:
    pickle.dump(final_dataset, output)